# B2B Lead Scoring: Data Exploration & Quality Assessment

**Objective:** Understand the lead dataset structure, assess data quality, and identify initial patterns that will inform our scoring model.

**Business Context:** We're analyzing leads from a B2B online education company to understand:
- What does our lead data look like?
- Are there data quality issues we need to address?
- What patterns exist between lead characteristics and conversion?
- Which features are most likely to be predictive?

## 1. Setup & Data Loading

In [ ]:
# Import libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings

# Configure display settings
warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 100)

# Set visualization style
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)

print("✓ Libraries loaded successfully")

In [ ]:
# Load the dataset
# Note: Download from https://www.kaggle.com/datasets/ashydv/leads-dataset

df = pd.read_csv('../data/Leads.csv')

print(f"Dataset loaded: {df.shape[0]:,} rows × {df.shape[1]} columns")
print(f"\nMemory usage: {df.memory_usage(deep=True).sum() / 1024**2:.2f} MB")

## 2. Initial Data Overview

In [ ]:
# First look at the data
print("=" * 80)
print("SAMPLE DATA (First 5 rows)")
print("=" * 80)
df.head()

In [ ]:
# Column information
print("=" * 80)
print("COLUMN INFORMATION")
print("=" * 80)
df.info()

In [ ]:
# Statistical summary
print("=" * 80)
print("NUMERICAL FEATURES - STATISTICAL SUMMARY")
print("=" * 80)
df.describe()

## 3. Target Variable Analysis

Understanding our conversion rate is critical - it tells us how balanced our dataset is.

In [ ]:
# Conversion rate analysis
target_column = 'Converted'  # Adjust if your dataset uses a different column name

conversion_counts = df[target_column].value_counts()
conversion_rate = df[target_column].mean() * 100

print("=" * 80)
print("CONVERSION ANALYSIS")
print("=" * 80)
print(f"\nTotal Leads: {len(df):,}")
print(f"Converted: {conversion_counts.get(1, 0):,}")
print(f"Not Converted: {conversion_counts.get(0, 0):,}")
print(f"\n📊 Overall Conversion Rate: {conversion_rate:.2f}%")

# Visualize
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Count plot
sns.countplot(data=df, x=target_column, ax=ax1, palette=['#e74c3c', '#2ecc71'])
ax1.set_title('Lead Conversion Distribution', fontsize=14, fontweight='bold')
ax1.set_xlabel('Converted (0=No, 1=Yes)', fontsize=12)
ax1.set_ylabel('Number of Leads', fontsize=12)

# Add count labels on bars
for p in ax1.patches:
    ax1.annotate(f'{int(p.get_height()):,}', 
                (p.get_x() + p.get_width() / 2., p.get_height()),
                ha='center', va='bottom', fontsize=11)

# Pie chart
colors = ['#e74c3c', '#2ecc71']
ax2.pie(conversion_counts, labels=['Not Converted', 'Converted'], 
        autopct='%1.1f%%', colors=colors, startangle=90)
ax2.set_title('Conversion Rate Breakdown', fontsize=14, fontweight='bold')

plt.tight_layout()
plt.savefig('../visualizations/conversion_distribution.png', dpi=300, bbox_inches='tight')
plt.show()

print("\n✓ Visualization saved to: visualizations/conversion_distribution.png")

## 4. Data Quality Assessment

**Critical for CRM data:** Missing values and data quality issues are common. Let's identify them.

In [ ]:
# Missing values analysis
missing_data = pd.DataFrame({
    'Column': df.columns,
    'Missing_Count': df.isnull().sum(),
    'Missing_Percentage': (df.isnull().sum() / len(df) * 100).round(2)
}).sort_values('Missing_Percentage', ascending=False)

missing_data = missing_data[missing_data['Missing_Count'] > 0]

print("=" * 80)
print("DATA QUALITY REPORT: Missing Values")
print("=" * 80)

if len(missing_data) > 0:
    print(f"\n⚠️  Found {len(missing_data)} columns with missing values:\n")
    print(missing_data.to_string(index=False))
    
    # Visualize missing data
    plt.figure(figsize=(12, 6))
    sns.barplot(data=missing_data.head(15), y='Column', x='Missing_Percentage', 
                palette='Reds_r')
    plt.title('Top 15 Columns with Missing Data', fontsize=14, fontweight='bold')
    plt.xlabel('Percentage Missing (%)', fontsize=12)
    plt.ylabel('Column Name', fontsize=12)
    plt.tight_layout()
    plt.savefig('../visualizations/missing_data_analysis.png', dpi=300, bbox_inches='tight')
    plt.show()
    
    print("\n✓ Visualization saved to: visualizations/missing_data_analysis.png")
else:
    print("\n✓ No missing values found - excellent data quality!")

In [ ]:
# Check for duplicate records
duplicates = df.duplicated().sum()
print(f"\nDuplicate records: {duplicates:,}")

if duplicates > 0:
    print(f"⚠️  Found {duplicates} duplicate rows - will need to handle these")
else:
    print("✓ No duplicate records found")

## 5. Feature Type Categorization

Let's categorize our features to understand what we're working with.

In [ ]:
# Identify numerical vs categorical features
numerical_features = df.select_dtypes(include=['int64', 'float64']).columns.tolist()
categorical_features = df.select_dtypes(include=['object']).columns.tolist()

# Remove target from feature lists if present
if target_column in numerical_features:
    numerical_features.remove(target_column)

print("=" * 80)
print("FEATURE CATEGORIZATION")
print("=" * 80)
print(f"\n📊 Numerical Features ({len(numerical_features)}):")
for i, feat in enumerate(numerical_features, 1):
    print(f"  {i}. {feat}")

print(f"\n📝 Categorical Features ({len(categorical_features)}):")
for i, feat in enumerate(categorical_features, 1):
    print(f"  {i}. {feat}")

## 6. Exploratory Data Analysis - Key Features

Let's examine some critical features that typically drive conversion in B2B.

In [ ]:
# Analyze Lead Source performance
# This is typically one of the most important features in B2B marketing

if 'Lead Source' in df.columns:
    lead_source_analysis = df.groupby('Lead Source').agg({
        target_column: ['count', 'sum', 'mean']
    }).round(3)
    
    lead_source_analysis.columns = ['Total_Leads', 'Conversions', 'Conversion_Rate']
    lead_source_analysis = lead_source_analysis.sort_values('Conversion_Rate', ascending=False)
    lead_source_analysis['Conversion_Rate_Pct'] = (lead_source_analysis['Conversion_Rate'] * 100).round(2)
    
    print("=" * 80)
    print("LEAD SOURCE PERFORMANCE")
    print("=" * 80)
    print(lead_source_analysis.head(10))
    
    # Visualize top sources
    top_sources = lead_source_analysis.nlargest(8, 'Total_Leads')
    
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))
    
    # Volume by source
    top_sources['Total_Leads'].plot(kind='barh', ax=ax1, color='steelblue')
    ax1.set_title('Top 8 Lead Sources by Volume', fontsize=14, fontweight='bold')
    ax1.set_xlabel('Number of Leads', fontsize=12)
    
    # Conversion rate by source
    top_sources['Conversion_Rate_Pct'].plot(kind='barh', ax=ax2, color='seagreen')
    ax2.set_title('Conversion Rate by Lead Source', fontsize=14, fontweight='bold')
    ax2.set_xlabel('Conversion Rate (%)', fontsize=12)
    
    plt.tight_layout()
    plt.savefig('../visualizations/lead_source_performance.png', dpi=300, bbox_inches='tight')
    plt.show()
    
    print("\n✓ Visualization saved")

In [ ]:
# Analyze numerical features - behavioral signals
# Features like Total Visits, Time on Site are key engagement indicators

behavioral_features = ['TotalVisits', 'Total Time Spent on Website', 'Page Views Per Visit']
available_behavioral = [f for f in behavioral_features if f in df.columns]

if available_behavioral:
    print("=" * 80)
    print("BEHAVIORAL ENGAGEMENT ANALYSIS")
    print("=" * 80)
    
    # Compare converted vs non-converted
    comparison_df = df.groupby(target_column)[available_behavioral].mean()
    print("\nAverage Engagement Metrics by Conversion Status:")
    print(comparison_df)
    
    # Visualize
    fig, axes = plt.subplots(1, len(available_behavioral), figsize=(16, 5))
    if len(available_behavioral) == 1:
        axes = [axes]
    
    for i, feature in enumerate(available_behavioral):
        df.boxplot(column=feature, by=target_column, ax=axes[i])
        axes[i].set_title(f'{feature}\nby Conversion Status', fontsize=11)
        axes[i].set_xlabel('Converted (0=No, 1=Yes)')
        axes[i].set_ylabel(feature)
    
    plt.suptitle('Behavioral Engagement: Converted vs Non-Converted', 
                fontsize=14, fontweight='bold', y=1.02)
    plt.tight_layout()
    plt.savefig('../visualizations/behavioral_analysis.png', dpi=300, bbox_inches='tight')
    plt.show()
    
    print("\n✓ Visualization saved")

## 7. Correlation Analysis

Understanding which features correlate with conversion helps us prioritize.

In [ ]:
# Correlation with target variable
if numerical_features:
    correlations = df[numerical_features + [target_column]].corr()[target_column].sort_values(ascending=False)
    correlations = correlations.drop(target_column)  # Remove self-correlation
    
    print("=" * 80)
    print("FEATURE CORRELATION WITH CONVERSION")
    print("=" * 80)
    print("\nTop Positive Correlations (features that increase with conversion):")
    print(correlations.head(10))
    
    print("\nTop Negative Correlations (features that decrease with conversion):")
    print(correlations.tail(5))
    
    # Visualize top correlations
    plt.figure(figsize=(10, 8))
    top_corr = pd.concat([correlations.head(10), correlations.tail(5)])
    colors = ['green' if x > 0 else 'red' for x in top_corr]
    
    top_corr.plot(kind='barh', color=colors)
    plt.title('Feature Correlation with Lead Conversion', fontsize=14, fontweight='bold')
    plt.xlabel('Correlation Coefficient', fontsize=12)
    plt.axvline(x=0, color='black', linestyle='--', linewidth=0.8)
    plt.tight_layout()
    plt.savefig('../visualizations/feature_correlations.png', dpi=300, bbox_inches='tight')
    plt.show()
    
    print("\n✓ Visualization saved")

## 8. Key Insights Summary

Let's document what we've learned.

In [ ]:
print("=" * 80)
print("KEY INSIGHTS FROM DATA EXPLORATION")
print("=" * 80)

insights = f"""
📊 DATASET OVERVIEW:
- Total Leads: {len(df):,}
- Features: {len(df.columns)}
- Conversion Rate: {conversion_rate:.2f}%

🔍 DATA QUALITY:
- Columns with missing data: {len(missing_data) if len(missing_data) > 0 else 'None - excellent!'}
- Duplicate records: {duplicates:,}

💡 INITIAL PATTERNS:
- Feature types: {len(numerical_features)} numerical, {len(categorical_features)} categorical
- Class balance: {'Balanced' if 40 <= conversion_rate <= 60 else 'Imbalanced - will need to consider in modeling'}

🎯 NEXT STEPS:
1. Handle missing values and data quality issues
2. Feature engineering to create behavioral scores
3. Build and evaluate lead scoring model
4. Generate business insights and recommendations
"""

print(insights)

## 9. Save Processed Data for Next Steps

We'll save a checkpoint of our initial exploration.

In [ ]:
# Save exploration results
exploration_summary = {
    'total_leads': len(df),
    'conversion_rate': conversion_rate,
    'numerical_features': numerical_features,
    'categorical_features': categorical_features,
    'missing_columns': missing_data['Column'].tolist() if len(missing_data) > 0 else []
}

# This will be used in subsequent notebooks
import json
with open('../data/exploration_summary.json', 'w') as f:
    json.dump(exploration_summary, f, indent=2)

print("✓ Exploration summary saved to: data/exploration_summary.json")
print("\n🎉 Data exploration complete! Ready for model building.")